## LLMLingua2

<a target="_blank" href="https://colab.research.google.com/github/microsoft/LLMLingua/blob/main/examples/LLMLingua2.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

<a target="_blank" href="https://aclanthology.org/2024.findings-acl.57/">LLMLingua-2</a> focuses on task-agnostic prompt compression for better generalizability and efficiency. It is a small-size yet powerful prompt compression method trained via data distillation from GPT-4 for token classification with a BERT-level encoder, excels in <b>task-agnostic compression</b>. It surpasses LLMLingua in handling <b>out-of-domain data</b>, offering <b>3x-6x faster</b> performance.

Below, We showcase the usage and compression results of <i>LLMLingua-2</i> on both <b>in-domain</b> and <b>out-of-domain</b> datasets, including various tasks such as single-document QA, multi-document QA, summarization and in-context learning.


In [ ]:
from llmlingua import PromptCompressor

llm_lingua = PromptCompressor(
    model_name="microsoft/llmlingua-2-xlm-roberta-large-meetingbank",
    use_llmlingua2=True,
)

### Target LLM Config

In [ ]:
!pip install openai==0.28

In [ ]:
# Using the OAI
import openai

openai.api_key = "<insert_openai_key>"

# or Using the AOAI
import openai

openai.api_key = "<insert_openai_key>"
openai.api_base = "<insert_openai_base>"
openai.api_type = "azure"
openai.api_version = "2023-05-15"

## In-Domain

Below, we present the results of <i>LLMLingua-2</i> compared to the strong baselines on In-Domain data: test set of <a href="https://aclanthology.org/2023.acl-long.906/">MeetingBank</a>.
Despite the fact that our compressors are much smaller than the LLaMa-2-7B used in the baselines, 
our approach achieves <b>significantly better performance</b> on both the QA and Summary tasks, and <b>comes close to matching the performance of the original prompt</b>.

### MeetingBank


In [ ]:
# Download the original prompt and dataset
from datasets import load_dataset

dataset = load_dataset("huuuyeah/meetingbank", split="test")
context = dataset[0]["transcript"]

question = "What is the agenda item three resolution 31669 about?\nAnswer:"
reference = "Encouraging individualized tenant assessment."

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt = "\n\n".join([context, question])

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": 100,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    engine="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

In [ ]:
# 2000 Compression
compressed_prompt = llm_lingua.compress_prompt(
    context,
    rate=0.33,
    force_tokens=["!", ".", "?", "\n"],
    drop_consecutive=True,
)

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt = "\n\n".join([compressed_prompt["compressed_prompt"], question])

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": 100,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    model="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

## Out-of-Domain

As our model is only trained on meeting transcripts data from MeetingBank, here we explore its generalization ability across various benchmarks of long-context scenarios, reasoning, and in-context learning.
Although the compressor of <i>LLMLingua-2</i> is only trained on MeetingBank data, <i>LLMLingua-2</i> is also effective on <b>out-of-domain</b> data, 
with its performance <b>comparable to or even surpassing</b> the SOTA <i>task-agnostic</i> compression baselines. 

Below, we showcase several compression results on <a href="https://arxiv.org/abs/2308.14508">LongBench</a> and <a href="https://arxiv.org/abs/2110.14168">GSM8K</a>, including single-document QA, multi-document QA, summarization and in-context learning tasks.

### Load LongBench Prompt

In [ ]:
dataset2prompt = {
    "narrativeqa": "You are given a story, which can be either a novel or a movie script, and a question. Answer the question asconcisely as you can, using a single phrase if possible. Do not provide any explanation.\n\nStory: {context}\n\nNow, answer the question based on the story asconcisely as you can, using a single phrase if possible. Do not provide any explanation.\n\nQuestion: {input}\n\nAnswer:",
    "gov_report": "You are given a report by a government agency. Write a one-page summary of the report.\n\nReport:\n{context}\n\nNow, write a one-page summary of the report.\n\nSummary:",
    "triviaqa": "Answer the question based on the given passage. Only give me the answer and do not output any other words. The following are some examples.\n\n{context}\n\n{input}",
}

dataset2maxlen = {
    "narrativeqa": 128,
    "gov_report": 512,
    "triviaqa": 32,
}

### Single-Doc QA

In [ ]:
task = "narrativeqa"
dataset = load_dataset("THUDM/LongBench", task, split="test")
sample = dataset[3]
context = sample["context"]
reference = sample["answers"]
print(reference)

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt_format = dataset2prompt[task]
max_gen = int(dataset2maxlen[task])
prompt = prompt_format.format(**sample)

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": max_gen,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    engine="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

In [ ]:
# 3000 Compression
compressed_prompt = llm_lingua.compress_prompt(
    context,
    target_token=3000,
    force_tokens=["!", ".", "?", "\n"],
    drop_consecutive=True,
)

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt_format = dataset2prompt[task]
max_gen = int(dataset2maxlen[task])
sample["context"] = compressed_prompt["compressed_prompt"]
prompt = prompt_format.format(**sample)

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": max_gen,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    engine="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

### Multi-Doc QA

In [ ]:
task = "triviaqa"
dataset = load_dataset("THUDM/LongBench", task, split="test")
sample = dataset[0]
context = sample["context"]
reference = sample["answers"]
print(reference)

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt_format = dataset2prompt[task]
max_gen = int(dataset2maxlen[task])
prompt = prompt_format.format(**sample)

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": max_gen,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    engine="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

In [ ]:
context_list = context.split("\nPassage:")
context_list = ["\nPassage:" + c for c in context_list]

# 2000 Compression
compressed_prompt = llm_lingua.compress_prompt(
    context_list,
    target_token=2000,
    force_tokens=["\nPassage:", ".", "?", "\n"],
    drop_consecutive=True,
    use_context_level_filter=True,
)

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt_format = dataset2prompt[task]
max_gen = int(dataset2maxlen[task])
sample["context"] = compressed_prompt["compressed_prompt"]
prompt = prompt_format.format(**sample)

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": max_gen,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    engine="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

### Summarization

In [ ]:
task = "gov_report"
dataset = load_dataset("THUDM/LongBench", task, split="test")
sample = dataset[0]
context = sample["context"]
reference = sample["answers"]
print(reference)

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt_format = dataset2prompt[task]
max_gen = int(dataset2maxlen[task])
prompt = prompt_format.format(**sample)

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": max_gen,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    engine="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

In [ ]:
# 3000 Compression
compressed_prompt = llm_lingua.compress_prompt(
    context,
    target_token=3000,
    force_tokens=["!", ".", "?", "\n"],
    drop_consecutive=True,
)

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt_format = dataset2prompt[task]
max_gen = int(dataset2maxlen[task])
sample["context"] = compressed_prompt["compressed_prompt"]
prompt = prompt_format.format(**sample)

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": max_gen,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    engine="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

### In-Context Learning (GSM8K)

In [ ]:
!wget https://raw.githubusercontent.com/FranxYao/chain-of-thought-hub/main/gsm8k/lib_prompt/prompt_hardest.txt
prompt_complex = open("./prompt_hardest.txt").read()
gsm8k = load_dataset("gsm8k", "main")
gsm8k_test = gsm8k["test"]

In [ ]:
# select an example from GSM8K
question, answer = [gsm8k_test[2][key] for key in ["question", "answer"]]
# Ground-truth Answer
print("Question:", question)
print("Answer:", answer)

In [ ]:
# The response from original prompt
import json

instruction = "Please reference the following examples to answer the math question,\n"
prompt = instruction + prompt_complex + "\n\nQuestion: " + question

request_data = {
    "prompt": prompt,
    "max_tokens": 400,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
    "stop": "\n\n",
}
response = openai.Completion.create(
    engine="gpt-35-turbo-instruct",
    **request_data,
)
print(json.dumps(response, indent=4))

In [ ]:
# 3000 Compression
compressed_prompt = llm_lingua.compress_prompt(
    prompt_complex.split("\n\n"),
    target_token=150,
    force_tokens=["+", "-", "*", "×", "/", "÷", "=", "The answer is", "\n"],
    drop_consecutive=True,
    force_reserve_digit=True,
    use_context_level_filter=True,
)

In [ ]:
instruction = "Please reference the following examples to answer the math question,\n"
prompt = (
    instruction + compressed_prompt["compressed_prompt"] + "\n\nQuestion: " + question
)

request_data = {
    "prompt": prompt,
    "max_tokens": 400,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
    "stop": "\r\n",
}
response = openai.Completion.create(
    engine="gpt-35-turbo-instruct",
    **request_data,
)
print("Response:", response)